# Demo 01: Catching a Model in the Act

Week 4, Module 01. Attention, induction heads, and why attention is not an explanation.

Instructor demo (we-do). We build this together, live. Where you see a WE-DO banner in a cell, that is a line we write as a room.

## The retail hook

A Cordwell Home and Hardware support assistant often has to repeat back something it saw earlier in the same prompt: an order number, a SKU, a customer name. That copy-it-from-earlier behavior is the same mechanism that makes few-shot prompting and context echoing work. Today we find the circuit inside GPT-2 that does this copying, and we prove that it is the one responsible.

## Engineer framing

Two ideas map onto tools you already trust.

- An attention pattern is a trace. It shows where each token looked when it built its representation. Think of it like a flame graph for one forward pass.
- Ablation is a feature flag. We turn one component off and measure what breaks. If the behavior collapses, that component was causal.

## What you will be able to do

1. Load GPT-2 small with TransformerLens and read its internal activations.
2. Show in-context copying with a live loss number.
3. Visualize the attention pattern of a single head.
4. Score every head for induction behavior, a correlational signal.
5. Ablate the top head and measure the causal jump in loss.
6. Explain why a high attention weight is not, by itself, an explanation.

Time budget: about 40 minutes. Natural break after Part 3. The appendix is optional.

## Part 0: Setup

We load GPT-2 small through TransformerBridge, the current TransformerLens load path. The older HookedTransformer.from_pretrained entry point is deprecated as of TransformerLens 3.0 and is being removed in the next major version, so all new code uses the bridge.

Two setup choices worth stating out loud.

- Device. The cohort machines are Macs with no GPU, so this runs on CPU. GPT-2 small is tiny, so CPU is fast. The device is selected at runtime, never hard-coded. TransformerLens does not auto-select Apple MPS because MPS can silently return wrong attention numbers on some PyTorch builds, and this demo reads attention numbers directly, so CPU is the safe default here. See the README if you want to opt into MPS.
- Compatibility mode. TransformerBridge keeps raw Hugging Face weights by default. We call enable_compatibility_mode so the model uses the classic TransformerLens numerics (folded LayerNorm, centered weights) and exposes the legacy hook names this demo relies on, such as hook_z and the pattern cache alias. Cross-entropy loss is unaffected by this call.

Instructor note. GPT-2 small weights download from Hugging Face on first load, so run Part 0 once on a connected machine before class. See the README for the offline fallback.

In [ ]:
# If TransformerLens is not installed yet, uncomment the next line.
# %pip install -q transformer-lens==3.5.1 matplotlib pandas

import os
import random
from functools import partial

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformer_lens.model_bridge import TransformerBridge

# Reproducibility. Fixed seeds so the demo looks the same every run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def pick_device() -> str:
    """Select a device at runtime. Never hard-code cuda.

    Order is cuda, then Apple MPS, then cpu. MPS is only chosen when the
    caller has opted in with TRANSFORMERLENS_ALLOW_MPS=1, because MPS can
    return silently wrong attention on some torch builds and this demo
    reads attention values directly. On the cohort's no-GPU Macs this
    returns cpu, which is fast for a model this small.
    """
    try:
        if torch.cuda.is_available():
            return "cuda"
        mps = getattr(torch.backends, "mps", None)
        if mps is not None and mps.is_available() and mps.is_built():
            if os.environ.get("TRANSFORMERLENS_ALLOW_MPS", "") == "1":
                return "mps"
    except Exception:
        pass
    return "cpu"


# Config variables. Never hard-code a model id inline.
MODEL_NAME = "gpt2"        # official GPT-2 small checkpoint
DEVICE = pick_device()

# Load through the bridge, then switch on classic TransformerLens numerics
# and legacy hook names (hook_z, pattern alias, and so on).
model = TransformerBridge.boot_transformers(MODEL_NAME, device=DEVICE)
model.enable_compatibility_mode()
model.eval()

# Facts we reuse throughout.
meta = {
    "model": MODEL_NAME,
    "n_layers": model.cfg.n_layers,   # 12
    "n_heads": model.cfg.n_heads,     # 12 heads per layer, 144 total
    "d_model": model.cfg.d_model,     # 768
    "d_vocab": model.cfg.d_vocab,     # 50257
    "device": DEVICE,
    "seed": SEED,
}
meta

## Part 1: The phenomenon, in-context copying

We feed the model a block of random tokens, then repeat the exact same block. The model has never seen these random tokens together before, so it cannot have memorized them. If it predicts the second copy well, that is pure in-context copying.

We measure prediction quality with next-token loss (cross entropy). Lower loss means a better prediction.

Prediction to make with the room before we run it. Will loss on the second copy be higher or lower than on the first copy?

In [ ]:
def make_repeated_tokens(model, seq_len=50, batch=4):
    """Build [BOS, r, r] where r is a block of random tokens, repeated once.

    Returns a tensor of shape [batch, 2*seq_len + 1]. The second copy is
    identical to the first, so predicting it only requires copying from
    earlier in the same sequence.
    """
    bos = model.tokenizer.bos_token_id
    rand = torch.randint(1000, model.cfg.d_vocab - 1, (batch, seq_len), device=DEVICE)
    bos_col = torch.full((batch, 1), bos, dtype=torch.long, device=DEVICE)
    # ===================== WE-DO =====================
    # Build [BOS, r, r]: the bos column, then the random block, then the
    # same random block again, concatenated along the position axis (dim=1).
    # This repeat is the whole experiment.
    return ...   # replace ... together
    # =================================================

SEQ_LEN = 50
tokens = make_repeated_tokens(model, seq_len=SEQ_LEN, batch=4)
print("token block shape:", tuple(tokens.shape))   # expect [4, 101]

# Per-token loss. Shape [batch, pos - 1]. Entry i is the loss predicting token i + 1.
with torch.no_grad():
    loss_per_token = model(tokens, return_type="loss", loss_per_token=True)

# ===================== WE-DO =====================
# first_copy_loss: mean loss over the first block, positions [:, :SEQ_LEN].
# second_copy_loss: mean loss over the second block AFTER its first token,
#   positions [:, SEQ_LEN + 1:]. We skip that one boundary token because it
#   has no earlier match to copy from. Take .mean().item() on each.
first_copy_loss = ...    # replace ... together
second_copy_loss = ...   # replace ... together
# =================================================

print(f"first copy mean loss:  {first_copy_loss:.3f}")
print(f"second copy mean loss: {second_copy_loss:.3f}")
print(f"drop from copying:     {first_copy_loss - second_copy_loss:.3f}")

The second copy is much easier for the model even though the tokens are random. Nothing about the tokens themselves is predictable. The only new information is that they already appeared once. The model is copying from earlier in the prompt. This behavior is called induction, and the heads that implement it are induction heads (Olsson et al. 2022).

This is the mechanism behind context echoing. When the Cordwell assistant repeats an order number back correctly, induction-style copying is a large part of why it can.

## Part 2: Look inside, the correlational view

Now we open the model up. For one sequence we cache every attention pattern, then score each head for how much it behaves like an induction head.

The induction signal. In the second copy, a head doing induction attends from the current token back to the token that came right after the previous copy of the current token. On a repeated block, that target sits on a fixed diagonal of the attention matrix. We read that diagonal and average it. A high score means the head consistently looks at the copy target.

In [ ]:
# One sequence, cache all activations. GPT-2 small is small, so this is cheap.
tokens_one = make_repeated_tokens(model, seq_len=SEQ_LEN, batch=1)
with torch.no_grad():
    _, cache = model.run_with_cache(tokens_one)

def induction_score_per_head(cache, n_layers, n_heads, seq_len):
    """Average attention along the induction diagonal for every head.

    cache["pattern", layer] has shape [batch, head, query_pos, key_pos].
    For a repeated block of length seq_len the induction target sits on the
    diagonal with offset (1 - seq_len): key = query + 1 - seq_len.
    """
    scores = torch.zeros((n_layers, n_heads))
    for layer in range(n_layers):
        pattern = cache["pattern", layer]                                  # [1, head, q, k]
        # ===================== WE-DO =====================
        # Pull the induction diagonal from pattern and average it per head.
        #   1. pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len) gives
        #      [1, head, d]: the diagonal where key = query + 1 - seq_len.
        #   2. mean over that last axis, then take batch index 0, then .cpu().
        stripe = ...                       # replace ... together
        scores[layer] = ...                # replace ... together
        # =================================================
    return scores

scores = induction_score_per_head(cache, model.cfg.n_layers, model.cfg.n_heads, SEQ_LEN)

# Rank heads by induction score.
flat = [(scores[l, h].item(), l, h)
        for l in range(model.cfg.n_layers)
        for h in range(model.cfg.n_heads)]
top_heads = sorted(flat, reverse=True)[:5]
print("Top induction heads (score, layer, head):")
for s, l, h in top_heads:
    print(f"  L{l} H{h}: {s:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(scores.numpy(), aspect="auto", cmap="viridis")
ax.set_xlabel("head")
ax.set_ylabel("layer")
ax.set_title("Induction score per head (brighter means more induction-like)")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
fig.colorbar(im, label="mean attention on the induction diagonal")
plt.tight_layout()
plt.show()

A few bright cells stand out in the middle-to-later layers. Those are our induction head candidates. Everything else barely registers on this signal. This is a map of where to look next, not proof of anything yet.

In [ ]:
# Draw the attention pattern of the single strongest head this run.
top_layer, top_head = top_heads[0][1], top_heads[0][2]
pattern = cache["pattern", top_layer][0, top_head].cpu().numpy()   # [query, key]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(pattern, aspect="auto", cmap="viridis")
ax.set_title(f"Attention pattern, L{top_layer} H{top_head}")
ax.set_xlabel("key position (looked at)")
ax.set_ylabel("query position (doing the looking)")
fig.colorbar(im, label="attention weight")
plt.tight_layout()
plt.show()

print(f"Top induction head this run: L{top_layer} H{top_head}")

Look at the second half of the sequence, the rows below the halfway point. There is a bright diagonal offset from the main diagonal. Each bright cell is the current token looking back at the token that followed the previous copy of itself. That is the induction move, drawn as a picture.

This is still only a correlation. A bright stripe tells us where the head looks. It does not tell us whether the head actually causes the copying, or whether the model would fail without it. For that we have to intervene.

## Part 3: Prove it, the causal view

We turn the top head off and measure what happens to the second-copy loss. Turning a head off means zeroing its output for this forward pass, so it contributes nothing to the residual stream. If the head really implements copying, the second-copy loss should jump.

We zero the head output at hook_z, the per-head result just before it is written back into the residual stream. Zero-ablation is the simplest intervention to explain. A more rigorous variant is mean-ablation, which replaces the head output with its average instead of zero.

In [ ]:
def zero_head_hook(z, hook, head):
    # z has shape [batch, pos, head, d_head]. Zero one head, leave the rest.
    # ===================== WE-DO =====================
    # Set this head's slice of z to 0.0 across all batches and positions,
    # then return z. Index is [:, :, head, :].
    ...   # replace ... together
    # =================================================
    return z

def second_copy_loss_ablated(tokens, layer, head, seq_len):
    hook_name = f"blocks.{layer}.attn.hook_z"
    fn = partial(zero_head_hook, head=head)
    # ===================== WE-DO =====================
    # Attach fn at hook_name for one forward pass using model.hooks(...),
    # then run the model with return_type="loss", loss_per_token=True and
    # read the second-copy slice [:, seq_len + 1:].mean().item().
    with model.hooks(fwd_hooks=[(hook_name, fn)]):
        with torch.no_grad():
            lpt = model(tokens, return_type="loss", loss_per_token=True)
    # =================================================
    return lpt[:, seq_len + 1:].mean().item()

# Baseline second-copy loss, nothing ablated.
with torch.no_grad():
    base_lpt = model(tokens, return_type="loss", loss_per_token=True)
baseline = base_lpt[:, SEQ_LEN + 1:].mean().item()

ablated_one = second_copy_loss_ablated(tokens, top_layer, top_head, SEQ_LEN)

print(f"second-copy loss, baseline:         {baseline:.3f}")
print(f"second-copy loss, top head off:     {ablated_one:.3f}")
print(f"loss increase from removing 1 head: {ablated_one - baseline:.3f}")

In [ ]:
def second_copy_loss_ablate_many(tokens, heads, seq_len):
    fwd_hooks = []
    for layer, head in heads:
        hook_name = f"blocks.{layer}.attn.hook_z"
        fwd_hooks.append((hook_name, partial(zero_head_hook, head=head)))
    with model.hooks(fwd_hooks=fwd_hooks):
        with torch.no_grad():
            lpt = model(tokens, return_type="loss", loss_per_token=True)
    return lpt[:, seq_len + 1:].mean().item()

# The five highest-scoring heads, turned off together.
top_k = [(l, h) for (_, l, h) in top_heads]
ablated_many = second_copy_loss_ablate_many(tokens, top_k, SEQ_LEN)

print("Ablating the top 5 induction heads together:")
print(f"  baseline second-copy loss: {baseline:.3f}")
print(f"  ablated second-copy loss:  {ablated_many:.3f}")
print(f"  loss increase:             {ablated_many - baseline:.3f}")

Removing a small number of heads sends the second-copy loss back up toward the no-context level. The copying behavior lived largely in these few heads. That is a causal claim, and we earned it with an intervention, not with a heatmap.

This is the memorable moment. A handful of the 144 heads carry most of the in-context copying, and we can switch it off and watch it break.

## Part 4: Attention is not an explanation

Here is the trap. It is tempting to look at a bright attention pattern and say the model did something because it attended to some token. Attention weights are seductive because they are easy to plot. But they only show where a head reads. They say nothing about what the head writes into the residual stream, or whether that write helps.

We can see the gap directly. Rank the heads two ways: by induction attention score, and by how much loss increases when we ablate them. If attention told the whole story, the two rankings would match exactly.

In [ ]:
# Ablate each of the top candidate heads one at a time and record the effect.
candidates = [(l, h) for (_, l, h) in sorted(flat, reverse=True)[:8]]
rows = []
for layer, head in candidates:
    # ===================== WE-DO =====================
    # For each candidate head, measure its causal effect by ablating it alone.
    # Reuse second_copy_loss_ablated(tokens, layer, head, SEQ_LEN).
    loss_off = ...   # replace ... together
    # =================================================
    rows.append({
        "layer": layer,
        "head": head,
        "induction_score": round(scores[layer, head].item(), 3),
        "loss_increase_when_ablated": round(loss_off - baseline, 3),
    })

# Order by attention score so we can compare against the causal ordering.
table = (pd.DataFrame(rows)
         .sort_values("induction_score", ascending=False)
         .reset_index(drop=True))
table

The head with the highest attention score is usually not the head with the single largest causal effect, and the two orderings do not line up cleanly. Attention pointed us at the right neighborhood, but it did not rank the true contributors. Some heads attend strongly yet write something the model does not use here. Others share the work.

Why this happens. The residual stream is a running sum. Every head and every MLP adds a vector to it. Attention decides which earlier positions a head reads from. It does not decide what direction the head writes, and it does not decide whether other components reinforce or cancel that write. To make a claim about cause you have to intervene and measure, which is exactly what ablation does.

Responsible AI takeaway. When you build model evaluations or audits, do not ship an attention heatmap as your explanation of a decision. It is a hypothesis, not evidence. If a claim about why a model did something matters, back it with an intervention: ablate, patch, or otherwise change the input and measure the effect on the output.

## Recap

- Induction is in-context copying. We saw it as a large drop in loss on a repeated random block.
- Attention patterns are traces. We used them to find candidate heads and to draw the induction stripe.
- Ablation is a feature flag. Turning a few heads off broke the copying, which made the causal case.
- Attention is not explanation. Where a head looks is not the same as what it does or whether it matters.

Further reading, all verified. Olsson et al. 2022 on induction heads and in-context learning. Elhage et al. 2021 for the residual stream and the mathematical framework for transformer circuits. Wang et al. 2022 for name mover heads in the indirect object identification circuit.

## Appendix, optional: activation patching

Run this only if time allows. Ablation asks what breaks if this head is off. Patching asks the sharper question: is this specific head enough to carry the signal. We take the head output from a clean run and paste it into a run where the copy target is missing, then check whether the correct prediction comes back.

Setup. The clean sequence is a block repeated once, so induction works. The corrupt sequence uses a different first block, so the second copy has no earlier match and induction has nothing to copy. Both sequences share the exact same second half, so head outputs line up position by position.

In [ ]:
def make_block(model, seq_len, seed):
    g = torch.Generator(device="cpu").manual_seed(seed)
    block = torch.randint(1000, model.cfg.d_vocab - 1, (1, seq_len), generator=g)
    return block.to(DEVICE)

s = SEQ_LEN
block_a = make_block(model, s, 1)
block_b = make_block(model, s, 2)
bos_col = torch.full((1, 1), model.tokenizer.bos_token_id, dtype=torch.long, device=DEVICE)

clean = torch.cat([bos_col, block_a, block_a], dim=1)     # induction works
corrupt = torch.cat([bos_col, block_b, block_a], dim=1)   # copy target missing

# Cache the clean run, keeping only the top head layer z to stay light.
target_hook = f"blocks.{top_layer}.attn.hook_z"
with torch.no_grad():
    _, clean_cache = model.run_with_cache(
        clean, names_filter=lambda name: name == target_hook
    )
clean_z = clean_cache["z", top_layer]     # [1, pos, head, d_head]

def patch_head_hook(z, hook, head, source):
    # ===================== WE-DO =====================
    # Overwrite this head's slice of z with the same head's slice from the
    # clean run. Move source to z.device first. Index is [:, :, head, :].
    z[:, :, head, :] = ...   # replace ... together
    # =================================================
    return z

def measure_second_copy(tokens):
    with torch.no_grad():
        lpt = model(tokens, return_type="loss", loss_per_token=True)
    return lpt[:, s + 1:].mean().item()

clean_loss = measure_second_copy(clean)
corrupt_loss = measure_second_copy(corrupt)

with model.hooks(fwd_hooks=[(target_hook, partial(patch_head_hook, head=top_head, source=clean_z))]):
    with torch.no_grad():
        patched_lpt = model(corrupt, return_type="loss", loss_per_token=True)
patched_loss = patched_lpt[:, s + 1:].mean().item()

print(f"clean second-copy loss:         {clean_loss:.3f}")
print(f"corrupt second-copy loss:       {corrupt_loss:.3f}")
print(f"corrupt plus patched top head:  {patched_loss:.3f}")
print(f"recovery from one patched head: {corrupt_loss - patched_loss:.3f}")

If the patched loss falls well below the corrupt loss and toward the clean loss, then this one head, carrying its clean output, was enough to bring back most of the copying. Patching is a sharper test than ablation. Ablation shows a head is needed. Patching shows a head is sufficient for the piece we moved.

## Pre-flight self-check

Run this last, before class, on a connected machine. It confirms the three teaching beats actually hold on this run. If any assertion fails, investigate before you teach rather than in front of the room. In the student notebook, this cell only passes once every WE-DO line above has been filled in.

In [ ]:
# Beat 1: in-context copying exists. Second copy is much easier than the first.
assert first_copy_loss > second_copy_loss + 1.0, \
    "Induction drop too small. Check the repeated-sequence build."

# Beat 2: a real induction head was found.
assert top_heads[0][0] > 0.3, \
    "No strong induction head found. Check the score diagonal offset."

# Beat 3: removing the top heads breaks copying.
assert (ablated_many - baseline) > 0.3, \
    "Ablation did not raise loss. Check the hook_z ablation."

print("Pre-flight checks passed.")
print("The three beats hold: copying exists, a few heads implement it, removing them breaks it.")